# DS 208 · Week 8 — Data visualisation
## matplotlib and seaborn

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 7 — Charts on these slides draw in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 9 — A figure contains axes, and data are drawn on the axes


**Exercise.**

1. Predict what the x-axis tick labels will show.
2. Run the cell.
3. Change ax.plot to ax.bar and run again.


In [ ]:
import matplotlib.pyplot as plt

years = [2021, 2022, 2023, 2024, 2025]
sales = [12, 15, 14, 20, 24]

fig, ax = plt.subplots()
ax.plot(years, sales)
ax.set_title("Sales over time")
plt.show()

### Slide 11 — The flood data, loaded from the course API


Expected output:

```
(34079, 16)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
resp = requests.get(url, timeout=60)
df   = pd.read_csv(io.StringIO(resp.text))

# week 6's derived column: days from start to completion
for col in ["start_date", "completion_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["duration"] = (df.completion_date - df.start_date).dt.days

print(df.shape)

### Slide 12 — One call returns both objects, named


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df.budget, df.duration, alpha=0.05)
ax.set_xlabel("Budget (PHP)")
ax.set_ylabel("Duration (days)")
plt.show()

### Slide 14 — subplots returns an array for more than one axes


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8),
                         sharey=True)

axes[0, 0].hist(df.budget, bins=30)
axes[0, 1].hist(df.duration, bins=30)

fig.tight_layout()   # stops overlap
plt.show()

### Slide 16 — Overplotting with 27,664 dots


In [ ]:
x, y = df.budget, df.duration

# 1. transparency and small dots, on a log axis
fig, ax = plt.subplots()
ax.scatter(x, y, alpha=0.05, s=4)
ax.set_xscale("log")
plt.show()

# 2. a reproducible sample
d = df.sample(2000, random_state=42)

# 3. bin the plane (hexbin's own log axis)
pos = x > 0
fig, ax = plt.subplots()
ax.hexbin(x[pos], y[pos], gridsize=40, cmap="Blues", xscale="log")
plt.show()

### Slide 19 — An unlabelled chart cannot be interpreted


**Exercise.**

1. Predict what ax.set_xticks(years) changes.
2. Run the cell.
3. Delete the set_title line and run again. What can a reader no longer tell?


In [ ]:
# years, sales: the first chart's data
fig, ax = plt.subplots()
ax.plot(years, sales)
ax.set_xticks(years)
ax.set_xlabel("Year")
ax.set_ylabel("Sales (PHP thousands)")
ax.set_title("Cebu branch, 2021-2025")
plt.show()

### Slide 22 — Week 7’s eight cities, with their areas


In [ ]:
import io

csv_text = """city,region,pop
Quezon City,Luzon,2960000
Manila,Luzon,1847000
Baguio,Luzon,366000
Cebu,Visayas,964000
Iloilo,Visayas,457000
Tacloban,Visayas,251000
Davao,Mindanao,1777000
Cagayan de Oro,Mindanao,728000"""
cities = pd.read_csv(io.StringIO(csv_text))
areas = pd.DataFrame({
    "city": ["Quezon City", "Manila", "Baguio", "Cebu",
             "Iloilo", "Tacloban", "Davao", "Cagayan de Oro"],
    "area_km2": [172, 42, 58, 315, 78, 202, 2444, 413]})

both = cities.merge(areas, on="city")
by_region = both.groupby("region")["pop"].sum()

### Slide 23 — Line for time, bar for categories


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.plot(years, sales, marker="o")             # trend over time
a2.bar(by_region.index, by_region.values)     # compare categories
plt.show()

### Slide 24 — Spending per year in one call


Expected output:

```
year
2024    369.3
2025    196.2
2026      0.0
Name: budget, dtype: float64
```


In [ ]:
y = df.groupby("year").budget.sum() / 1e9

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(y.index, y.values, marker="o")
ax.set_ylabel("Total budget (PHP billions)")
plt.show()

print(y.round(1).tail(3))

### Slide 27 — An incomplete final year resembles a sharp decline


Expected output:

```
year
2024    5553
2025    3475
2026       1
dtype: int64
```


In [ ]:
print(df.groupby("year").size().tail(3))

# 1. drop what is not comparable
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(y[y.index <= 2024], marker="o")
plt.show()

# 2. or mark it: whole years solid, partial dashed
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(y[y.index <= 2024], marker="o")
ax.plot(y[y.index >= 2024], linestyle="--", color="grey")
ax.annotate("partial year", xy=(2025, y[2025]), xytext=(2025.3, 250),
            arrowprops=dict(arrowstyle="->"))
plt.show()

### Slide 29 — Sorted, horizontal, and starting at zero


Expected output:

```
region
Region V                   155.6
National Capital Region    159.2
Region III                 267.7
Name: budget, dtype: float64
```


In [ ]:
g = (df.groupby("region").budget.sum()
       .sort_values() / 1e9)

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(g.index, g.values)
ax.set_xlabel("Total budget (PHP billions)")
plt.show()

print(g.round(1).tail(3))

### Slide 31 — When a second y-axis is misleading


In [ ]:
totals = df.groupby("year").budget.sum() / 1e9
counts = df.groupby("year").size()

# tempting, and easy to abuse
fig, ax = plt.subplots()
ax.plot(totals.index, totals.values)
ax2 = ax.twinx()
ax2.plot(counts.index, counts.values, color="red")
plt.show()

# better: two panels, one shared year axis
fig, (a1, a2) = plt.subplots(2, 1, sharex=True)
a1.plot(totals.index, totals.values)
a2.bar(counts.index, counts.values)
plt.show()

### Slide 33 — Scatter for pairs, histogram for one column


**Exercise.**

1. Predict whether cities with a larger area have more people.
2. Run the cell.
3. Change bins=5 to bins=2. What does the histogram hide now?


In [ ]:
# both: cities with their areas (the tables slide)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.scatter(both["area_km2"], both["pop"])   # do they move together?
a2.hist(both["pop"], bins=5)                # where do values cluster?
plt.show()

### Slide 37 — Passing a DataFrame to seaborn


In [ ]:
import seaborn as sns

sns.barplot(data=both, x="region", y="pop")
plt.show()
sns.scatterplot(data=both, x="area_km2", y="pop")
plt.show()

### Slide 39 — Axes-level and figure-level functions


In [ ]:
import seaborn as sns

# AXES-level: give it an ax
fig, ax = plt.subplots()
sns.scatterplot(data=both, x="area_km2", y="pop", ax=ax)
plt.show()

### Slide 39 — Axes-level and figure-level functions


In [ ]:
import seaborn as sns

# FIGURE-level: it owns the figure, one panel per region
g = sns.relplot(data=both, x="area_km2", y="pop", col="region")
plt.show()

### Slide 40 — Figure-level for facets, axes-level for composed layouts


In [ ]:
# d: the 2,000-project sample from the overplotting slide
g = sns.relplot(data=d, x="budget", y="duration",
                col="region", col_wrap=6,
                height=2.2, alpha=0.3)
g.set(xscale="log")
g.savefig("by_region.png", dpi=150)   # g.savefig, not plt.savefig
plt.show()

### Slide 43 — Legends only for multiple series


In [ ]:
t = df[df.year <= 2025].pivot_table(index="year", columns="region",
                                     values="budget", aggfunc="sum") / 1e9

def draw(ax):
    ax.plot(t.index, t["Region III"], label="Region III")
    ax.plot(t.index, t["National Capital Region"], label="NCR")

fig, ax = plt.subplots()
draw(ax)
ax.legend(loc="upper left", frameon=False)
plt.show()

# outside, saved WITHOUT bbox_inches="tight"
fig, ax = plt.subplots()
draw(ax)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.savefig("legend_outside.png", dpi=150)

### Slide 45 — Style set once at the top of the notebook


In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid",
              palette="colorblind",
              context="talk")
# context scales ALL text at once:
#   paper < notebook < talk < poster

### Slide 48 — A truncated bar axis multiplies differences


Expected output:

```
region
Region IX     0.656
Region XII    0.920
Name: status, dtype: float64
```


In [ ]:
rates = ((df.status == "Completed")
           .groupby(df.region).mean().sort_values())
print(rates.round(3).iloc[[0, -1]])

for top in [1, 0.95]:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(rates.index, rates.values)
    ax.set_ylim(0 if top == 1 else 0.6, top)   # full, then truncated
    plt.show()

### Slide 50 — Readers assume a linear scale unless told otherwise


In [ ]:
fig, ax = plt.subplots()
ax.scatter(df.budget, df.duration, alpha=0.05, s=4)
ax.set_xscale("log")
ax.set_xlabel("Budget (PHP, log scale)")
ax.set_ylabel("Duration (days)")
plt.show()

### Slide 52 — Colour choice for meaning and accessibility


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

grid = np.arange(12).reshape(3, 4)        # 0 to 11

fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3))
a1.imshow(grid, cmap="viridis")           # sequential: low to high
a2.imshow(grid - 5.5, cmap="RdBu_r")      # diverging: centred on 0
plt.show()

### Slide 53 — Writing the message on the chart


In [ ]:
d = df.dropna(subset=["duration"])          # projects with a duration
q1, q3 = d.budget.quantile([.25, .75])
top = d[d.budget >= q3].duration.median()
low = d[d.budget <= q1].duration.median()
co = df[df.region == "Central Office"]
x0, y0 = co.budget.median(), co.duration.median()

fig, ax = plt.subplots()
ax.scatter(df.budget, df.duration, alpha=0.05, s=4)
ax.set_xscale("log")
ax.annotate("Central Office: 180 projects,\nmedian PHP 144.5M",
            xy=(x0, y0), xytext=(x0 * 2, y0 + 500),
            arrowprops=dict(arrowstyle="->"))
ax.set_title(f"Larger projects take longer: {top:.0f} vs {low:.0f} median days")
plt.show()

### Slide 55 — Adjusting the layout and saving at high resolution


In [ ]:
fig, ax = plt.subplots()
ax.bar(by_region.index, by_region.values)
fig.tight_layout()
fig.savefig("pops.png", dpi=150)
plt.show()   # display in the notebook

### Slide 56 — Two arguments that prevent cropped labels


In [ ]:
fig, ax = plt.subplots()
ax.scatter(both["area_km2"], both["pop"])
fig.savefig("area_pop.png", dpi=150, bbox_inches="tight")

# vector, for a report that gets zoomed
fig.savefig("area_pop.pdf", bbox_inches="tight")

# in a script, close what you save
plt.close(fig)

### Slide 61 — Population by region: a complete chart


**Exercise.**

1. Predict which region’s bar is longest, and roughly its share.
2. Run the cell.
3. Remove .sort_values() and run again. What becomes harder to see?


In [ ]:
by_reg = cities.groupby("region")["pop"].sum().sort_values()
share = by_reg / by_reg.sum()
print(share.round(3))

ax = by_reg.plot.barh()
ax.set_xlabel("Population of the eight cities")
ax.set_title(f"Luzon's three cities hold {share['Luzon']:.0%} of the people")
plt.show()

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
